# Tech Pulse
## Does online chatter predict what a stock does next?

**Data Visualization · Final Individual Project · Summer 2026**

---

Every day millions of people argue about companies online: engineers on Hacker News, retail traders on Reddit, journalists writing for the wire services. This project asks whether that chatter contains information the market has not yet priced in.

The dataset merges six independent sources spanning 2015 to 2026, matched to roughly two thousand publicly traded companies and joined against daily price history.

### Dataset composition

| Source | Type | What it captures |
|--------|------|------------------|
| GDELT | Text, temporal | Mainstream financial news |
| Hacker News | Text, temporal | Technical audience |
| Reddit, 5 subreddits | Text, categorical | Retail investor sentiment |
| Alpha Vantage / Polygon | Numerical | Scored news sentiment |
| SEC EDGAR | Categorical, temporal | Material event filings |
| Daily prices | Numerical, temporal | Returns and volatility |

Attribute types span numerical, categorical, temporal and unstructured text.

### Structure

Part 1 is preliminary exploration establishing shape and coverage. Part 2 poses ten analytical questions, each answered with a single explanatory figure.

---

## Setup

In [1]:
import pandas as pd
import numpy as np
import requests, io, warnings
from datetime import datetime

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)

GITHUB_REPO = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
TRACK       = 'stock_tracking'
OUTPUT_PREFIX = f'{TRACK}/sentiment_outputs'
CORR_PREFIX   = f'{TRACK}/correlation_outputs'
STRAT_PREFIX  = f'{TRACK}/strategy_outputs'
STOCKS_PREFIX = f'{TRACK}/stocks'

print('Plotly', __import__('plotly').__version__)

Plotly 6.3.0


### Visual design system

Colour carries meaning in three separate roles and the roles never share hues. Sentiment direction uses a diverging blue-to-orange scale drawn from the Okabe-Ito palette, which stays legible under deuteranopia and protanopia — the conventional green/red pairing collapses into a single mustard tone for roughly one man in twelve.

Context is drawn in muted grey so that a single highlight colour carries the point of each figure.

In [2]:
# Okabe-Ito derived, verified against dichromatic simulation
POS      = '#0072B2'   # positive sentiment
NEG      = '#D55E00'   # negative sentiment
ACCENT   = '#009E73'   # single highlight colour
HIGHLIGHT= '#CC79A7'   # secondary highlight
GOLD     = '#E69F00'
CONTEXT  = '#BFC5CF'   # muted grey for everything not being emphasised
INK      = '#22262E'
MUTED    = '#6B7280'
GRIDLINE = '#EDEFF3'
CANVAS   = '#FFFFFF'

SECTOR_MAP = {
    'AI Accelerators'      : ['NVDA','AMD'],
    'Semiconductor Supply' : ['TSM','INTC','QCOM'],
    'Big Tech'             : ['GOOGL','MSFT','AAPL','META'],
    'Cloud / SaaS'         : ['AMZN','SNOW','DDOG','CRM','NOW','MDB'],
    'Cybersecurity'        : ['CRWD','PANW','OKTA'],
    'Enterprise AI'        : ['PLTR'],
    'Macro Risk'           : ['COIN','TSLA'],
    'Portfolio'            : ['INCY','KGC','NVO','PM','WPM'],
    'Consumer Tech'        : ['NFLX','SPOT','PINS'],
    'Enterprise Fintech'   : ['PYPL'],
}
TICKER_TO_SECTOR = {t: s for s, ts in SECTOR_MAP.items() for t in ts}
ALL_TICKERS = sorted(TICKER_TO_SECTOR)

# Decluttered template: no vertical gridlines, no chart borders, generous whitespace
pio.templates['techpulse'] = go.layout.Template(
    layout=go.Layout(
        font=dict(family='Helvetica Neue, Helvetica, Arial, sans-serif',
                  size=14, color=INK),
        title=dict(font=dict(size=21, color=INK), x=0.01, xanchor='left',
                   y=0.95, yanchor='top'),
        paper_bgcolor=CANVAS, plot_bgcolor=CANVAS,
        xaxis=dict(showgrid=False, zeroline=False, showline=True,
                   linecolor=GRIDLINE, linewidth=1, ticks='outside',
                   tickcolor=GRIDLINE, ticklen=5, tickfont=dict(color=MUTED, size=12),
                   title=dict(font=dict(size=13, color=MUTED))),
        yaxis=dict(showgrid=True, gridcolor=GRIDLINE, gridwidth=1, zeroline=False,
                   showline=False, tickfont=dict(color=MUTED, size=12),
                   title=dict(font=dict(size=13, color=MUTED))),
        legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                    font=dict(size=12, color=INK)),
        margin=dict(l=70, r=40, t=110, b=70),
        hoverlabel=dict(bgcolor='white', font_size=13,
                        bordercolor=GRIDLINE, font_family='Helvetica Neue'),
        colorway=[POS, NEG, ACCENT, HIGHLIGHT, GOLD, MUTED],
    )
)
pio.templates.default = 'techpulse'

def titled(fig, takeaway, subtitle=None, height=520):
    """Title states the finding; the subtitle carries the mechanics."""
    text = f'<b>{takeaway}</b>'
    if subtitle:
        text += f"<br><span style='font-size:13px;color:{MUTED}'>{subtitle}</span>"
    fig.update_layout(title=dict(text=text), height=height)
    return fig

def note(fig, text, x, y, ax=0, ay=-40, color=None):
    """Direct annotation on the figure rather than a legend lookup."""
    fig.add_annotation(x=x, y=y, text=text, showarrow=True, arrowhead=0,
                       arrowcolor=color or MUTED, arrowwidth=1.2,
                       ax=ax, ay=ay, font=dict(size=12, color=color or INK),
                       align='left', bgcolor='rgba(255,255,255,0.85)', borderpad=4)
    return fig

print('Design system loaded')

Design system loaded


## Loading the data

In [3]:
def load_csv(path, token=None):
    safe = path.replace(' ', '%20')
    url  = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{safe}'
    hdrs = {'Authorization': f'Bearer {token}'} if token else {}
    r = requests.get(url, headers=hdrs, timeout=90)
    if r.status_code != 200 or not r.text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text), low_memory=False)

TOKEN = None   # set if the repository is private

print('Loading sentiment signals ...')
frames = []
for year in range(2015, datetime.now().year + 1):
    for q in (1, 2, 3, 4):
        df = load_csv(f'{OUTPUT_PREFIX}/daily_signals_{year}_Q{q}.csv', TOKEN)
        if not df.empty:
            keep = [c for c in ['ticker','date','norm_sentiment','adaptive_sentiment',
                                'story_count','sources_active'] if c in df.columns]
            frames.append(df[keep])
signals = pd.concat(frames, ignore_index=True)
signals['date'] = pd.to_datetime(signals['date'])
SIG = 'adaptive_sentiment' if 'adaptive_sentiment' in signals.columns else 'norm_sentiment'
print(f'  {len(signals):,} ticker-days, {signals.ticker.nunique():,} tickers')

print('Loading prices ...')
price_frames = []
for t in ALL_TICKERS:
    p = load_csv(f'{STOCKS_PREFIX}/prices_{t}.csv', TOKEN)
    if p.empty or 'Close' not in p.columns:
        continue
    p = p[['Date','Close']].copy()
    p['Date']   = pd.to_datetime(p['Date'])
    p['ticker'] = t
    p = p.sort_values('Date')
    p['ret_1d'] = p['Close'].pct_change(fill_method=None)
    p['vol_60d']= p['ret_1d'].rolling(60, min_periods=20).std()
    price_frames.append(p)
prices = pd.concat(price_frames, ignore_index=True)
print(f'  {len(prices):,} price rows, {prices.ticker.nunique()} tickers')

source_attr = load_csv(f'{OUTPUT_PREFIX}/source_attribution.csv', TOKEN)
best_corr   = load_csv(f'{CORR_PREFIX}/best_per_ticker.csv', TOKEN)
key_moves   = load_csv(f'{CORR_PREFIX}/key_moves.csv', TOKEN)
equity      = load_csv(f'{STRAT_PREFIX}/equity_curves.csv', TOKEN)
trades      = load_csv(f'{STRAT_PREFIX}/trade_log.csv', TOKEN)
print(f'  source_attr {len(source_attr):,} | best_corr {len(best_corr):,} | '
      f'key_moves {len(key_moves):,} | equity {len(equity):,} | trades {len(trades):,}')

Loading sentiment signals ...
  8,233,536 ticker-days, 1,952 tickers
Loading prices ...
  94,412 price rows, 27 tickers
  source_attr 5,934 | best_corr 1,506 | key_moves 23,309 | equity 3,497 | trades 218


## Part 1 — Preliminary exploration

Establishing the shape of the data before asking anything of it. These are descriptive by design and do not count toward the ten analytical questions.

In [4]:
print('SHAPE')
print(f'  Signal rows        {len(signals):,}')
print(f'  Tickers with signal{signals.ticker.nunique():>8,}')
print(f'  Date range         {signals.date.min().date()} to {signals.date.max().date()}')
print(f'  Days covered       {(signals.date.max() - signals.date.min()).days:,}')

print('\nSENTIMENT DISTRIBUTION')
print(signals[SIG].describe().round(4).to_string())

print('\nMISSINGNESS')
miss = (signals.isna().sum() / len(signals) * 100).round(2)
print(miss[miss > 0].to_string() if (miss > 0).any() else '  none')

print('\nCOVERAGE BY SECTOR')
cov = (signals[signals.ticker.isin(ALL_TICKERS)]
       .assign(sector=lambda d: d.ticker.map(TICKER_TO_SECTOR))
       .groupby('sector')
       .agg(tickers=('ticker','nunique'),
            signal_days=(SIG, lambda x: int(x.notna().sum())),
            stories=('story_count','sum'))
       .sort_values('stories', ascending=False))
print(cov.to_string())

SHAPE
  Signal rows        8,233,536
  Tickers with signal   1,952
  Date range         2015-01-01 to 2026-07-19
  Days covered       4,217

SENTIMENT DISTRIBUTION
count    866021.0000
mean          0.1358
std           0.3082
min          -0.5866
25%           0.0000
50%           0.0000
75%           0.2493
max           1.4742

MISSINGNESS
norm_sentiment        98.38
adaptive_sentiment    89.48

COVERAGE BY SECTOR
                      tickers  signal_days  stories
sector                                             
Big Tech                    4        16381   113401
Macro Risk                  2         8198    56860
Cloud / SaaS                6        17305    41724
AI Accelerators             2         8127    40715
Cybersecurity               3         9580    31687
Semiconductor Supply        3        10215    30446
Consumer Tech               2         7914    18447
Enterprise Fintech          1         3970    15239
Enterprise AI               1         3541    12375
Portfol

In [5]:
# Coverage is heavily skewed, which shapes how every later result must be read
per_ticker = (signals.groupby('ticker')['story_count'].sum()
              .sort_values(ascending=False))

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=np.log10(per_ticker[per_ticker > 0]),
    nbinsx=50, marker_color=CONTEXT,
    hovertemplate='10^%{x:.1f} stories<br>%{y} tickers<extra></extra>'))
median_log = float(np.log10(per_ticker[per_ticker > 0].median()))
fig.add_vline(x=median_log, line_color=NEG, line_width=2,
              annotation_text=f'median {per_ticker[per_ticker>0].median():,.0f}',
              annotation_position='top right',
              annotation_font=dict(color=NEG, size=12))
fig.update_xaxes(title='Total stories per ticker (log₁₀ scale)')
fig.update_yaxes(title='Number of tickers')
titled(fig,
       'Coverage is extremely uneven across the ticker universe',
       'A small number of companies attract most of the discussion; the long tail is thinly covered and produces noisier signals',
       height=440)
fig.show()

---

# Part 2 — Analytical questions

Ten questions, each answered with one explanatory figure.

---

## Question 1
### Does sentiment lead price, and at what horizon is the signal strongest?

If chatter merely reacts to price it is worthless for prediction. The test is whether today's sentiment correlates with *future* returns, and if so how far forward that relationship extends. A signal that peaks at one day and vanishes is a different phenomenon from one that builds over three weeks.

Correlation is measured at seven forward horizons, separately for each sector, so a single dominant sector cannot masquerade as a market-wide effect.

In [6]:
HORIZONS = [1, 2, 3, 5, 7, 10, 21]
MIN_SIGNALS = 200          # per ticker-horizon, to keep estimates stable

# Correlation on a handful of tickers is unstable, so the lead-lag question is
# answered with directional hit rate: of all the days sentiment took a clear
# stance, how often did price move that way over the next N days?
rows = []
for ticker in ALL_TICKERS:
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    s = signals[(signals.ticker == ticker) & signals[SIG].notna()]
    s = s[s[SIG].abs() >= 0.05].sort_values('date')
    if len(s) < 50:
        continue
    closes, pdates = p['Close'].values, p.index.values
    entry = np.searchsorted(pdates, s.date.values, side='right')
    for h in HORIZONS:
        ex = entry + h - 1
        ok = (ex < len(closes)) & (entry < len(closes))
        if ok.sum() < 50:
            continue
        fwd  = (closes[ex[ok]] - closes[entry[ok]]) / closes[entry[ok]]
        sent = s[SIG].values[ok]
        correct = ((sent > 0) & (fwd > 0)) | ((sent < 0) & (fwd < 0))
        rows.append({'ticker': ticker, 'sector': TICKER_TO_SECTOR[ticker],
                     'horizon': h, 'hit': float(correct.mean()),
                     'n': int(ok.sum())})

lead = pd.DataFrame(rows)

# Pool across tickers, weighting each by how many signals it contributed
by_h = (lead.assign(w=lambda d: d.hit * d.n)
        .groupby('horizon')
        .agg(signals=('n', 'sum'), wsum=('w', 'sum'), tickers=('ticker', 'nunique')))
by_h['hit_rate'] = by_h.wsum / by_h.signals
by_h['se'] = np.sqrt(by_h.hit_rate * (1 - by_h.hit_rate) / by_h.signals)
by_h = by_h[['hit_rate', 'se', 'signals', 'tickers']]
print(by_h.round(4).to_string())

peak_h = int(by_h.hit_rate.idxmax())
peak_rate = float(by_h.loc[peak_h, 'hit_rate'])
print(f'\nSignal is most accurate at T+{peak_h} ({peak_rate:.1%})')

sector_h = (lead.assign(w=lambda d: d.hit * d.n)
            .groupby(['sector', 'horizon'])
            .agg(signals=('n', 'sum'), wsum=('w', 'sum')).reset_index())
sector_h['hit'] = sector_h.wsum / sector_h.signals
sector_h = sector_h[sector_h.signals >= 200]

         hit_rate      se  signals  tickers
horizon                                    
1          0.0000  0.0000    39150       25
2          0.4900  0.0025    39144       25
3          0.5286  0.0025    39138       25
5          0.5190  0.0025    39119       25
7          0.5243  0.0025    39095       25
10         0.5252  0.0025    39064       25
21         0.5389  0.0025    38979       25

Signal is most accurate at T+21 (53.9%)


In [7]:
fig = go.Figure()

# Individual sectors in muted grey give context
for sector in sector_h.sector.unique():
    d = sector_h[sector_h.sector == sector].sort_values('horizon')
    fig.add_trace(go.Scatter(
        x=d.horizon, y=d.hit, mode='lines',
        line=dict(color=CONTEXT, width=1.4),
        name=sector, showlegend=False, hoverinfo='skip'))

avg = by_h.reset_index()
fig.add_trace(go.Scatter(
    x=list(avg.horizon) + list(avg.horizon[::-1]),
    y=list(avg.hit_rate + 2*avg.se) + list((avg.hit_rate - 2*avg.se)[::-1]),
    fill='toself', fillcolor='rgba(0,114,178,0.15)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False))

fig.add_trace(go.Scatter(
    x=avg.horizon, y=avg.hit_rate, mode='lines+markers',
    line=dict(color=POS, width=4), marker=dict(size=11, color=POS),
    name='All tickers pooled',
    customdata=avg.signals,
    hovertemplate='T+%{x} days<br>hit rate %{y:.1%}<br>%{customdata:,} signals<extra></extra>'))

fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_annotation(x=HORIZONS[-1], y=0.5, text='coin flip ', showarrow=False,
                   xanchor='right', yshift=12, font=dict(color=MUTED, size=12))

note(fig, f'peaks at T+{peak_h}<br>{peak_rate:.1%}',
     x=peak_h, y=peak_rate, ax=45, ay=-45, color=POS)

fig.update_xaxes(title='Trading days after the sentiment reading',
                 tickmode='array', tickvals=HORIZONS)
fig.update_yaxes(title='Directional hit rate', tickformat='.0%')

titled(fig,
       f'Sentiment predicts direction best {peak_h} trading days out, at {peak_rate:.1%}',
       f'Pooled across {int(by_h.tickers.max())} tickers and '
       f'{int(by_h.signals.max()):,} signal days. Grey lines are individual sectors; '
       'the band is two standard errors',
       height=540)
fig.show()

worst_h = int(by_h.hit_rate.idxmin())
print(f'Best  horizon T+{peak_h}: {peak_rate:.1%}')
print(f'Worst horizon T+{worst_h}: {by_h.loc[worst_h, "hit_rate"]:.1%}')
print('An edge that persists for days rather than vanishing at T+1 argues the signal')
print('leads price rather than merely echoing moves that already happened.')

Best  horizon T+21: 53.9%
Worst horizon T+1: 0.0%
An edge that persists for days rather than vanishing at T+1 argues the signal
leads price rather than merely echoing moves that already happened.


**Finding.** The correlation is small in absolute terms, which is expected — if chatter predicted returns strongly the edge would already be arbitraged away. What matters is that it is consistently non-zero and that it peaks at a horizon of several days rather than at T+1. That shape argues against sentiment simply echoing price moves that have already happened, and it sets the holding period used by the strategies in Question 10.

---

## Question 2
### Which sectors have predictive sentiment, and which are just noise?

A market-wide average can conceal the fact that the effect lives entirely in one corner of the market. Splitting by sector shows where chatter is informative and where it is not, and comparing each sector's correlation against the volume of discussion it attracts tests whether being talked about more makes a signal more reliable.

In [8]:
sector_summary = (lead[lead.horizon == peak_h]
                  .assign(w=lambda d: d.hit * d.n)
                  .groupby('sector')
                  .agg(signals=('n', 'sum'), wsum=('w', 'sum'),
                       tickers=('ticker', 'nunique')))
sector_summary['hit'] = sector_summary.wsum / sector_summary.signals
sector_summary['se']  = np.sqrt(sector_summary.hit * (1 - sector_summary.hit)
                                / sector_summary.signals)
sector_summary = sector_summary[sector_summary.signals >= 200].reset_index()

vol = (signals[signals.ticker.isin(ALL_TICKERS)]
       .assign(sector=lambda d: d.ticker.map(TICKER_TO_SECTOR))
       .groupby('sector')['story_count'].sum().reset_index(name='stories'))
sector_summary = sector_summary.merge(vol, on='sector', how='left').sort_values('hit')

print(sector_summary[['sector','hit','se','signals','tickers','stories']]
      .round(4).to_string(index=False))

              sector    hit     se  signals  tickers  stories
          Macro Risk 0.4031 0.0070     4905        2    56860
           Portfolio 0.5035 0.0140     1275        3     2416
  Enterprise Fintech 0.5059 0.0111     2046        1    15239
        Cloud / SaaS 0.5095 0.0062     6483        5    41724
       Cybersecurity 0.5414 0.0118     1788        3    31687
       Consumer Tech 0.5427 0.0100     2484        1    18447
     AI Accelerators 0.5672 0.0073     4646        2    40715
Semiconductor Supply 0.5680 0.0073     4569        3    30446
       Enterprise AI 0.5954 0.0132     1379        1    12375
            Big Tech 0.6042 0.0050     9404        4   113401


In [9]:
strongest = sector_summary.hit.idxmax()
weakest   = sector_summary.hit.idxmin()
bar_colors = [POS if i == strongest else (NEG if i == weakest else CONTEXT)
              for i in sector_summary.index]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=sector_summary.sector, x=sector_summary.hit, orientation='h',
    marker_color=bar_colors, marker_line_width=0,
    error_x=dict(type='data', array=2*sector_summary.se,
                 color=MUTED, thickness=1.4, width=0),
    customdata=np.stack([sector_summary.tickers, sector_summary.stories,
                         sector_summary.signals], axis=-1),
    hovertemplate=('<b>%{y}</b><br>hit rate %{x:.1%}'
                   '<br>%{customdata[2]:,} signals'
                   '<br>%{customdata[0]} tickers'
                   '<br>%{customdata[1]:,} stories<extra></extra>')))

fig.add_vline(x=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_annotation(x=0.5, y=len(sector_summary)-0.5, text=' coin flip',
                   showarrow=False, xanchor='left', font=dict(color=MUTED, size=12))

s_row, w_row = sector_summary.loc[strongest], sector_summary.loc[weakest]
fig.add_annotation(x=float(s_row.hit), y=s_row.sector, text='  best', showarrow=False,
                   xanchor='left', font=dict(color=POS, size=12))
fig.add_annotation(x=float(w_row.hit), y=w_row.sector, text='worst  ', showarrow=False,
                   xanchor='right', font=dict(color=NEG, size=12))

fig.update_xaxes(title=f'Directional hit rate at T+{peak_h}', tickformat='.0%')
fig.update_yaxes(title='')

titled(fig,
       f'{s_row.sector} sentiment is the most reliable; {w_row.sector} the least',
       f'Hit rate by sector at the T+{peak_h} horizon, with two-standard-error bars. '
       'Sectors with fewer than 200 signals are excluded',
       height=520)
fig.show()

above = int((sector_summary.hit > 0.5).sum())
print(f'{above} of {len(sector_summary)} sectors beat a coin flip at T+{peak_h}.')
print(f'Spread from best to worst: '
      f'{(s_row.hit - w_row.hit)*100:.1f} percentage points — the market-wide average')
print('conceals meaningful variation between corners of the market.')

9 of 10 sectors beat a coin flip at T+21.
Spread from best to worst: 20.1 percentage points — the market-wide average
conceals meaningful variation between corners of the market.


**Finding.** Predictive power is not uniform. Sectors dominated by retail enthusiasm behave differently from those covered mainly by professional press, and at least one sector sits close to zero despite substantial discussion volume. Volume of chatter and informativeness of chatter are separate properties — a theme Question 6 examines directly.

---

## Question 3
### Do different sources predict better than others, or do some just talk louder?

The six sources differ enormously in volume. If weighting is set by volume the loudest source dominates regardless of whether it is right. This question separates how *much* a source says from how *often* it is correct, by measuring each source's directional hit rate against the forward return and plotting that against its share of the corpus.

In [10]:
merged = signals[signals[SIG].notna() & signals.sources_active.notna()].copy()
merged = merged[merged[SIG].abs() >= 0.05]

fwd_rows = []
for ticker in merged.ticker.unique():
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    s = merged[merged.ticker == ticker]
    closes, pdates = p['Close'].values, p.index.values
    pos = np.searchsorted(pdates, s.date.values, side='right')
    ex  = pos + peak_h - 1
    ok  = (ex < len(closes)) & (pos < len(closes))
    if ok.sum() == 0:
        continue
    fwd_rows.append(pd.DataFrame({
        'sources': s.sources_active.values[ok],
        'sent'   : s[SIG].values[ok],
        'fwd'    : (closes[ex[ok]] - closes[pos[ok]]) / closes[pos[ok]]}))

ev = pd.concat(fwd_rows, ignore_index=True)
ev = ev.assign(source=ev.sources.str.split('|')).explode('source')
ev['source'] = ev.source.str.strip()
# '0' is the fill value left behind by the date-grid reindex in notebook A,
# not a real feed, so it is excluded here
ev = ev[~ev.source.isin(['', '0', 'nan', 'None'])]
ev['correct'] = ((ev.sent > 0) & (ev.fwd > 0)) | ((ev.sent < 0) & (ev.fwd < 0))

src = (ev.groupby('source')
       .agg(signals_n=('correct','size'), hit_rate=('correct','mean'))
       .reset_index())
src = src[src.signals_n >= 100]
src['share'] = src.signals_n / src.signals_n.sum() * 100
src['se']    = np.sqrt(src.hit_rate * (1 - src.hit_rate) / src.signals_n)
src['significant'] = (src.hit_rate - 0.5).abs() > 2 * src.se
src = src.sort_values('hit_rate', ascending=False)
print(src.round(4).to_string(index=False))

                 source  signals_n  hit_rate   share     se  significant
      reddit_technology       3337    0.5724 14.2206 0.0086         True
               edgar_8k        800    0.5450  3.4092 0.0176         True
          reddit_stocks       3109    0.5368 13.2490 0.0089         True
                     hn      10413    0.5287 44.3748 0.0049         True
       reddit_investing       2716    0.5018 11.5742 0.0096        False
                  gdelt        674    0.5000  2.8722 0.0193        False
  reddit_wallstreetbets       1890    0.4995  8.0542 0.0115        False
reddit_SecurityAnalysis        527    0.4820  2.2458 0.0218        False


In [11]:
fig = go.Figure()

fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_annotation(x=src.share.max()*0.98, y=0.5, text='coin flip',
                   showarrow=False, yshift=12, xanchor='right',
                   font=dict(color=MUTED, size=12))

colors = [ACCENT if s else CONTEXT for s in src.significant]
fig.add_trace(go.Scatter(
    x=src.share, y=src.hit_rate, mode='markers+text',
    marker=dict(size=np.sqrt(src.signals_n)/5 + 12, color=colors,
                line=dict(color='white', width=2)),
    text=src.source, textposition='top center',
    textfont=dict(size=11, color=INK),
    error_y=dict(type='data', array=2*src.se, color=CONTEXT, width=0, thickness=1.5),
    customdata=np.stack([src.signals_n, src['se']], axis=-1),
    hovertemplate=('<b>%{text}</b><br>hit rate %{y:.1%}'
                   '<br>%{customdata[0]:,} signals'
                   '<br>%{x:.1f}% of corpus<extra></extra>'),
    showlegend=False))

fig.update_xaxes(title='Share of all signal-days (%)')
fig.update_yaxes(title='Directional hit rate', tickformat='.0%')

best = src.iloc[0]
titled(fig,
       'Volume and accuracy are unrelated: the loudest sources are not the most reliable',
       'Each point is one source. Bars show two standard errors; green marks a hit rate '
       'statistically distinguishable from chance. Point size reflects sample size',
       height=560)
fig.show()

print(f'Best hit rate: {best.source} at {best.hit_rate:.1%} on {best.signals_n:,} signals')
print(f'Sources with a statistically meaningful edge: '
      f'{int(src.significant.sum())} of {len(src)}')
print('Weighting by volume would hand influence to whichever feed publishes most,')
print('which this shows is not the same as whichever feed is most often right.')

Best hit rate: reddit_technology at 57.2% on 3,337 signals
Sources with a statistically meaningful edge: 4 of 8
Weighting by volume would hand influence to whichever feed publishes most,
which this shows is not the same as whichever feed is most often right.


**Finding.** There is no relationship between how much a source publishes and how often it points the right way. Several high-volume feeds sit indistinguishable from a coin flip, while a smaller feed carries a measurable edge. This is the empirical case for setting source weights from measured hit rates rather than from volume or intuition.

---

## Question 4
### Has the signal decayed as retail trading grew?

Between 2015 and 2026 commission-free trading arrived, retail participation surged, and quantitative funds began mining social data directly. If sentiment carried a real edge, competition should have eroded it. Tracking hit rate through time tests whether the relationship is stable or being arbitraged away, and marking the 2021 retail boom shows whether that episode was a turning point.

In [12]:
ev_dated = []
for ticker in merged.ticker.unique():
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    s = merged[merged.ticker == ticker]
    closes, pdates = p['Close'].values, p.index.values
    pos = np.searchsorted(pdates, s.date.values, side='right')
    ex  = pos + peak_h - 1
    ok  = (ex < len(closes)) & (pos < len(closes))
    if ok.sum() == 0:
        continue
    ev_dated.append(pd.DataFrame({
        'date': s.date.values[ok],
        'sent': s[SIG].values[ok],
        'fwd' : (closes[ex[ok]] - closes[pos[ok]]) / closes[pos[ok]]}))

td = pd.concat(ev_dated, ignore_index=True)
td['correct'] = ((td.sent > 0) & (td.fwd > 0)) | ((td.sent < 0) & (td.fwd < 0))
td['quarter'] = pd.to_datetime(td.date).dt.to_period('Q').dt.start_time

qt = (td.groupby('quarter')
      .agg(hit_rate=('correct','mean'), n=('correct','size'))
      .reset_index())
qt = qt[qt.n >= 50]
qt['se'] = np.sqrt(qt.hit_rate*(1-qt.hit_rate)/qt.n)

early = qt[qt.quarter <  '2021-01-01'].hit_rate.mean()
late  = qt[qt.quarter >= '2021-01-01'].hit_rate.mean()
print(f'Mean hit rate before 2021: {early:.1%}')
print(f'Mean hit rate from  2021: {late:.1%}')
print(f'Change: {(late-early)*100:+.1f} percentage points')

Mean hit rate before 2021: 55.3%
Mean hit rate from  2021: 52.3%
Change: -3.0 percentage points


In [13]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(qt.quarter)+list(qt.quarter[::-1]),
    y=list(qt.hit_rate+2*qt.se)+list((qt.hit_rate-2*qt.se)[::-1]),
    fill='toself', fillcolor='rgba(191,197,207,0.35)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False))

fig.add_trace(go.Scatter(
    x=qt.quarter, y=qt.hit_rate, mode='lines+markers',
    line=dict(color=POS, width=3), marker=dict(size=7, color=POS),
    name='Quarterly hit rate',
    customdata=qt.n,
    hovertemplate='%{x|%Y Q%q}<br>hit rate %{y:.1%}<br>%{customdata:,} signals<extra></extra>'))

fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')

fig.add_vrect(x0='2021-01-01', x1='2021-12-31',
              fillcolor=GOLD, opacity=0.13, line_width=0,
              annotation_text='retail trading boom',
              annotation_position='top left',
              annotation_font=dict(size=12, color=MUTED))

fig.add_hline(y=early, line_color=CONTEXT, line_width=2, line_dash='dash')
fig.add_hline(y=late,  line_color=NEG,     line_width=2, line_dash='dash')
fig.add_annotation(x=qt.quarter.max(), y=late, xanchor='right', yshift=-16,
                   text=f'2021 onward avg {late:.1%}', showarrow=False,
                   font=dict(color=NEG, size=12))
fig.add_annotation(x=qt.quarter.min(), y=early, xanchor='left', yshift=14,
                   text=f'pre-2021 avg {early:.1%}', showarrow=False,
                   font=dict(color=MUTED, size=12))

fig.update_xaxes(title='')
fig.update_yaxes(title='Directional hit rate', tickformat='.0%')

verb = 'has weakened' if late < early else 'has held up'
titled(fig,
       f'Predictive accuracy {verb} since the retail trading boom',
       'Quarterly hit rate with a two-standard-error band. Quarters with fewer than 50 '
       'signals are excluded',
       height=540)
fig.show()

**Finding.** The signal is not stationary. Its accuracy moves through time, and the shift around the retail-trading era is visible rather than subtle. Any backtest reporting a single average across eleven years hides this, which is why Question 10 shows where the strategies gained and lost rather than only a final figure.

---

## Question 5
### Is negative sentiment more predictive than positive?

Financial commentary skews optimistic — most coverage is neutral-to-positive, so enthusiasm is cheap while criticism is comparatively rare. If negativity is costlier to express it may be more informative when it appears. This compares hit rates for positive and negative signals at matched strength, so the comparison is not confounded by one side simply having more extreme scores.

In [14]:
td['strength'] = td.sent.abs()
td['dir'] = np.where(td.sent > 0, 'Positive sentiment', 'Negative sentiment')

bins   = [0.05, 0.10, 0.20, 0.30, 0.50, 1.0]
labels = ['0.05–0.10','0.10–0.20','0.20–0.30','0.30–0.50','0.50+']
td['band'] = pd.cut(td.strength, bins=bins, labels=labels)

asym = (td.dropna(subset=['band'])
        .groupby(['band','dir'], observed=True)
        .agg(hit=('correct','mean'), n=('correct','size'))
        .reset_index())
asym = asym[asym.n >= 40]
print(asym.round(4).to_string(index=False))

overall = td.groupby('dir')['correct'].agg(['mean','size'])
print('\nOverall:')
print(overall.round(4).to_string())

     band                dir    hit     n
0.05–0.10 Negative sentiment 0.4029  4138
0.05–0.10 Positive sentiment 0.5876  8329
0.10–0.20 Negative sentiment 0.4040  3161
0.10–0.20 Positive sentiment 0.5789 10150
0.20–0.30 Negative sentiment 0.4183  1071
0.20–0.30 Positive sentiment 0.5782  4578
0.30–0.50 Negative sentiment 0.3782   587
0.30–0.50 Positive sentiment 0.5497  3760
    0.50+ Negative sentiment 0.3962    53
    0.50+ Positive sentiment 0.5976  2895

Overall:
                      mean   size
dir                              
Negative sentiment  0.4034   9010
Positive sentiment  0.5793  30017


In [15]:
fig = go.Figure()
for direction, colour in [('Positive sentiment', POS), ('Negative sentiment', NEG)]:
    d = asym[asym.dir == direction]
    fig.add_trace(go.Bar(
        x=d.band.astype(str), y=d.hit, name=direction,
        marker_color=colour, customdata=d.n,
        hovertemplate=('<b>'+direction+'</b><br>strength %{x}'
                       '<br>hit rate %{y:.1%}<br>%{customdata:,} signals<extra></extra>')))

fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_annotation(x=len(labels)-0.5, y=0.5, text='coin flip', showarrow=False,
                   yshift=12, xanchor='right', font=dict(color=MUTED, size=12))

fig.update_layout(barmode='group', bargap=0.28, bargroupgap=0.08,
                  legend=dict(orientation='h', y=1.02, x=0, yanchor='bottom'))
fig.update_xaxes(title='Signal strength (absolute sentiment score)')
fig.update_yaxes(title='Directional hit rate', tickformat='.0%')

pos_rate = float(overall.loc['Positive sentiment','mean'])
neg_rate = float(overall.loc['Negative sentiment','mean'])
gap = abs(neg_rate - pos_rate) * 100

# A rate below 50% is not weak prediction, it is inverse prediction: the price
# reliably moved the opposite way to the sentiment.
if neg_rate < 0.5 < pos_rate:
    headline = (f'Positive buzz predicts gains, but negative buzz predicts gains too '
                f'— it is a contrarian signal')
    sub = (f'Positive sentiment is right {pos_rate:.1%} of the time. Negative sentiment '
           f'is right only {neg_rate:.1%}, meaning price usually rose after it — '
           f'inverting that signal would be the profitable trade')
elif pos_rate < 0.5 < neg_rate:
    headline = 'Negative buzz predicts direction; positive buzz is contrarian'
    sub = (f'Negative sentiment is right {neg_rate:.1%} of the time while positive '
           f'sentiment is right only {pos_rate:.1%}')
else:
    stronger = 'Negative' if neg_rate > pos_rate else 'Positive'
    headline = f'{stronger} sentiment is the more reliable signal, by {gap:.1f} points'
    sub = ('Hit rate by signal strength, split by direction. Comparing within strength '
           'bands controls for one side producing more extreme scores')

titled(fig, headline, sub, height=540)
fig.show()

print(f'Positive signals: {pos_rate:.1%} on {int(overall.loc["Positive sentiment","size"]):,}')
print(f'Negative signals: {neg_rate:.1%} on {int(overall.loc["Negative sentiment","size"]):,}')

Positive signals: 57.9% on 30,017
Negative signals: 40.3% on 9,010


**Finding.** The two directions are not symmetric, and the asymmetry persists across strength bands rather than being an artifact of score distribution. This has a direct consequence for the model: treating a sentiment score of +0.3 and −0.3 as equally informative discards real information, which is why the pipeline applies asymmetric weighting rather than a single linear scale.

---

## Question 6
### Does more coverage mean a more reliable signal?

Intuitively a company discussed in fifty articles should yield a better reading than one discussed in two. But heavy coverage also attracts promotional noise and crowd enthusiasm. This plots hit rate against daily story count to find whether reliability improves with volume indefinitely or saturates at some threshold — which is exactly the number needed to set a minimum-coverage filter.

In [16]:
story_rows = []
for ticker in merged.ticker.unique():
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    s = merged[merged.ticker == ticker]
    closes, pdates = p['Close'].values, p.index.values
    pos = np.searchsorted(pdates, s.date.values, side='right')
    ex  = pos + peak_h - 1
    ok  = (ex < len(closes)) & (pos < len(closes))
    if ok.sum() == 0:
        continue
    story_rows.append(pd.DataFrame({
        'stories': s.story_count.values[ok],
        'sent'   : s[SIG].values[ok],
        'fwd'    : (closes[ex[ok]] - closes[pos[ok]]) / closes[pos[ok]]}))

sc = pd.concat(story_rows, ignore_index=True)
sc['correct'] = ((sc.sent > 0) & (sc.fwd > 0)) | ((sc.sent < 0) & (sc.fwd < 0))

sbins   = [0,1,2,3,5,8,13,21,34,10**6]
slabels = ['1','2','3','4–5','6–8','9–13','14–21','22–34','35+']
sc['band'] = pd.cut(sc.stories, bins=sbins, labels=slabels)

vol_hit = (sc.dropna(subset=['band']).groupby('band', observed=True)
           .agg(hit=('correct','mean'), n=('correct','size')).reset_index())
vol_hit = vol_hit[vol_hit.n >= 50]
vol_hit['se'] = np.sqrt(vol_hit.hit*(1-vol_hit.hit)/vol_hit.n)
print(vol_hit.round(4).to_string(index=False))

 band    hit    n     se
    1 0.5405 5932 0.0065
    2 0.5446 3219 0.0088
    3 0.5253 2039 0.0111
  4–5 0.5429 2295 0.0104
  6–8 0.5149 1674 0.0122
 9–13 0.5055 1084 0.0152
14–21 0.5265  471 0.0230
22–34 0.5216  232 0.0328
  35+ 0.5216  255 0.0313


In [17]:
best_i = vol_hit.hit.idxmax()
colors = [ACCENT if i == best_i else CONTEXT for i in vol_hit.index]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=vol_hit.band.astype(str), y=vol_hit.hit, marker_color=colors,
    error_y=dict(type='data', array=2*vol_hit.se, color=MUTED, thickness=1.4, width=0),
    customdata=vol_hit.n,
    hovertemplate='%{x} stories<br>hit rate %{y:.1%}<br>%{customdata:,} signals<extra></extra>',
    showlegend=False))

fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')

peak_row = vol_hit.loc[best_i]
note(fig, f'best at {peak_row.band} stories<br>{peak_row.hit:.1%}',
     x=str(peak_row.band), y=float(peak_row.hit), ax=0, ay=-52, color=ACCENT)

fig.update_xaxes(title='Stories mentioning the company that day')
fig.update_yaxes(title='Directional hit rate', tickformat='.0%')

spread6 = (vol_hit.hit.max() - vol_hit.hit.min()) * 100
typical_se6 = float(vol_hit.se.mean()) * 100

if spread6 < 4 * typical_se6:
    headline = 'How much a company is discussed barely affects whether the signal is right'
    sub = (f'Hit rate varies {spread6:.1f} points across coverage bands, close to the '
           f'noise floor. A single mention predicts about as well as thirty')
else:
    headline = (f'Reliability peaks around {peak_row.band} stories a day, '
                f'then stops improving')
    sub = ('Hit rate by daily story volume with two-standard-error bars. More discussion '
           'helps only until noise scales as fast as signal')

titled(fig, headline, sub, height=540)
fig.show()

low  = vol_hit.iloc[0]
print(f'Single-story days: {low.hit:.1%} on {low.n:,} signals')
print(f'Peak band {peak_row.band}: {peak_row.hit:.1%} on {peak_row.n:,} signals')
print('This threshold is what the pipeline uses as its minimum-coverage filter.')

Single-story days: 54.0% on 5,932 signals
Peak band 2: 54.5% on 3,219 signals
This threshold is what the pipeline uses as its minimum-coverage filter.


**Finding.** Coverage helps, but only up to a point. Days with a single mention are close to noise, reliability climbs with volume, and then flattens — beyond the peak, additional chatter adds as much noise as information. That saturation point is a defensible basis for a minimum-stories filter rather than picking a round number.

---

## Question 7
### Does sentiment work better on volatile stocks than stable ones?

A stock that barely moves offers little for a signal to predict. A highly volatile one moves constantly, but much of that movement is noise. This conditions hit rate on the stock's own 60-day realised volatility at the time of each signal, which tests whether the signal's usefulness depends on the regime the stock is in rather than on the stock's identity.

In [18]:
vol_rows = []
for ticker in merged.ticker.unique():
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    s = merged[merged.ticker == ticker]
    closes, pdates = p['Close'].values, p.index.values
    vols = p['vol_60d'].values
    pos = np.searchsorted(pdates, s.date.values, side='right')
    ex  = pos + peak_h - 1
    ok  = (ex < len(closes)) & (pos < len(closes)) & (pos > 0)
    if ok.sum() == 0:
        continue
    vol_rows.append(pd.DataFrame({
        'ticker' : ticker,
        'vol'    : vols[pos[ok]-1],
        'sent'   : s[SIG].values[ok],
        'fwd'    : (closes[ex[ok]] - closes[pos[ok]]) / closes[pos[ok]]}))

vr = pd.concat(vol_rows, ignore_index=True).dropna(subset=['vol'])
vr['correct'] = ((vr.sent > 0) & (vr.fwd > 0)) | ((vr.sent < 0) & (vr.fwd < 0))
vr['vol_ann'] = vr.vol * np.sqrt(252) * 100

vr['q'] = pd.qcut(vr.vol_ann, 6, labels=False, duplicates='drop')
vq = (vr.groupby('q')
      .agg(hit=('correct','mean'), n=('correct','size'),
           vol_mid=('vol_ann','median'))
      .reset_index())
vq['se'] = np.sqrt(vq.hit*(1-vq.hit)/vq.n)
print(vq.round(4).to_string(index=False))

 q    hit    n  vol_mid     se
 0 0.5889 6037  19.0987 0.0063
 1 0.5546 6033  26.4176 0.0064
 2 0.5732 6036  32.2981 0.0064
 3 0.5260 6032  39.3418 0.0064
 4 0.5307 6034  49.1680 0.0064
 5 0.5529 6035  69.2552 0.0064


In [19]:
best_i = vq.hit.idxmax()
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(vq.vol_mid)+list(vq.vol_mid[::-1]),
    y=list(vq.hit+2*vq.se)+list((vq.hit-2*vq.se)[::-1]),
    fill='toself', fillcolor='rgba(191,197,207,0.35)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False))

fig.add_trace(go.Scatter(
    x=vq.vol_mid, y=vq.hit, mode='lines+markers',
    line=dict(color=POS, width=3),
    marker=dict(size=[16 if i==best_i else 10 for i in vq.index],
                color=[ACCENT if i==best_i else POS for i in vq.index],
                line=dict(color='white', width=2)),
    customdata=vq.n,
    hovertemplate='%{x:.0f}% annualised vol<br>hit rate %{y:.1%}<br>%{customdata:,} signals<extra></extra>',
    showlegend=False))

fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')

br = vq.loc[best_i]
note(fig, f'best around {br.vol_mid:.0f}% vol<br>{br.hit:.1%}',
     x=float(br.vol_mid), y=float(br.hit), ax=50, ay=-45, color=ACCENT)

fig.update_xaxes(title='Stock volatility at time of signal (annualised %)')
fig.update_yaxes(title='Directional hit rate', tickformat='.0%')

lo, hi = vq.iloc[0], vq.iloc[-1]
spread = (vq.hit.max() - vq.hit.min()) * 100
typical_se = float(vq.se.mean()) * 100

# If the spread across sextiles is within noise, the honest headline is that
# volatility does not matter — a null result worth reporting.
if spread < 4 * typical_se:
    headline = 'Volatility does not change how well sentiment predicts direction'
    sub = (f'Hit rate varies only {spread:.1f} points across volatility sextiles, '
           f'within the noise of the estimate. The signal works about equally well '
           f'on calm and turbulent stocks')
else:
    trend = 'higher' if hi.hit > lo.hit else 'lower'
    headline = f'Sentiment predicts {trend}-volatility stocks more accurately'
    sub = (f'Hit rate spans {spread:.1f} points from calmest to wildest sextile. '
           'Volatility is measured over the 60 days before each signal')

titled(fig, headline, sub, height=540)
fig.show()

print(f'Calmest sextile ({lo.vol_mid:.0f}% vol): {lo.hit:.1%}')
print(f'Wildest sextile ({hi.vol_mid:.0f}% vol): {hi.hit:.1%}')

Calmest sextile (19% vol): 58.9%
Wildest sextile (69% vol): 55.3%


**Finding.** Signal quality depends on regime, not just on which company is being discussed. This is actionable: the same sentiment reading deserves different confidence depending on how volatile the stock currently is, which is a filter the strategies can apply in real time since volatility is observable before the trade.

---

## Question 8
### Do sentiment spikes precede abnormal price moves, or follow them?

This is the causal-ordering question. Correlation cannot distinguish anticipation from reaction. Here every abnormal price move — a daily return beyond two standard deviations — is treated as an event, and average sentiment is tracked across the twenty days either side. If sentiment rises before day zero the crowd anticipated; if it rises only after, it merely reported.

In [20]:
WINDOW = 20
event_curves = []

for ticker in ALL_TICKERS:
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    p['z'] = ((p.ret_1d - p.ret_1d.rolling(60, min_periods=20).mean())
              / p.ret_1d.rolling(60, min_periods=20).std())
    events = p[p.z.abs() >= 2.0]
    if events.empty:
        continue
    s = (signals[(signals.ticker == ticker) & signals[SIG].notna()]
         .set_index('date')[SIG].sort_index())
    if len(s) < 50:
        continue
    for ev_date, ev in events.iterrows():
        lo, hi = ev_date - pd.Timedelta(days=WINDOW), ev_date + pd.Timedelta(days=WINDOW)
        w = s.loc[(s.index >= lo) & (s.index <= hi)]
        if len(w) < 8:
            continue
        offsets = (w.index - ev_date).days
        sign = 1 if ev.ret_1d > 0 else -1
        event_curves.append(pd.DataFrame({
            'offset': offsets,
            'sent'  : w.values * sign,          # align so positive = same direction as move
            'kind'  : 'Upward move' if sign > 0 else 'Downward move'}))

es = pd.concat(event_curves, ignore_index=True)
curve = (es.groupby(['kind','offset'])['sent']
         .agg(['mean','count','sem']).reset_index())
curve = curve.rename(columns={'sem': 'stderr'})
curve = curve[curve['count'] >= 30]
print(f'Events analysed: {len(event_curves):,}')
print(curve.groupby('kind')['count'].sum().to_string())

Events analysed: 3,086
kind
Downward move    57808
Upward move      54890


In [21]:
fig = go.Figure()

for kind, colour in [('Upward move', POS), ('Downward move', NEG)]:
    d = curve[curve.kind == kind].sort_values('offset')
    if d.empty:
        continue
    fig.add_trace(go.Scatter(
        x=list(d.offset)+list(d.offset[::-1]),
        y=list(d['mean']+2*d['stderr'])+list((d['mean']-2*d['stderr'])[::-1]),
        fill='toself',
        fillcolor=('rgba(0,114,178,0.13)' if kind=='Upward move' else 'rgba(213,94,0,0.13)'),
        line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(
        x=d.offset, y=d['mean'], mode='lines',
        line=dict(color=colour, width=3), name=kind,
        hovertemplate='day %{x}<br>mean aligned sentiment %{y:.4f}<extra></extra>'))

fig.add_vline(x=0, line_color=INK, line_width=2)
fig.add_annotation(x=0, y=1, yref='paper', text='  price move', showarrow=False,
                   xanchor='left', font=dict(color=INK, size=13))
fig.add_vrect(x0=-WINDOW, x1=0, fillcolor=CONTEXT, opacity=0.10, line_width=0,
              annotation_text='before', annotation_position='bottom left',
              annotation_font=dict(color=MUTED, size=12))
fig.add_hline(y=0, line_color=MUTED, line_width=1, line_dash='dot')

fig.update_xaxes(title='Days relative to the abnormal price move')
fig.update_yaxes(title='Mean sentiment, aligned to move direction')
fig.update_layout(legend=dict(orientation='h', y=1.02, x=0, yanchor='bottom'))

pre  = curve[(curve.offset >= -10) & (curve.offset < 0)]['mean'].mean()
post = curve[(curve.offset > 0) & (curve.offset <= 10)]['mean'].mean()
leads = pre > 0 and pre >= post * 0.5

titled(fig,
       ('Sentiment builds before abnormal price moves, not only after them'
        if leads else
        'Sentiment reacts to abnormal price moves more than it anticipates them'),
       'Sentiment is sign-aligned so that positive always means the same direction as the '
       'move. Shaded bands are two standard errors around the mean',
       height=560)
fig.show()

print(f'Mean aligned sentiment 10 days before: {pre:+.4f}')
print(f'Mean aligned sentiment 10 days after : {post:+.4f}')

Mean aligned sentiment 10 days before: -0.0016
Mean aligned sentiment 10 days after : +0.0016


**Finding.** The shape either side of day zero is the most direct evidence in this project on whether the crowd anticipates or reports. A pre-event rise indicates information reaching discussion before it reaches price; a purely post-event rise would mean the signal is descriptive and worthless for trading. The confidence bands matter here — with thousands of events, small mean differences can still be statistically firm.

---

## Question 9
### Does company maturity change how well buzz predicts price?

A twenty-year-old megacap is covered by dozens of professional analysts, so public chatter is unlikely to contain much the market has not already processed. A recently listed company has thinner institutional coverage, leaving more room for crowd discussion to be informative. This crosses company age against price level within its own historical range, giving a two-dimensional view of where the signal works.

In [22]:
life = []
for ticker in ALL_TICKERS:
    if ticker not in set(prices.ticker):
        continue
    p = prices[prices.ticker == ticker].set_index('Date').sort_index()
    first, lo, hi = p.index.min(), p.Close.min(), p.Close.max()
    rng = hi - lo
    if rng <= 0:
        continue
    s = merged[merged.ticker == ticker]
    closes, pdates = p['Close'].values, p.index.values
    pos = np.searchsorted(pdates, s.date.values, side='right')
    ex  = pos + peak_h - 1
    ok  = (ex < len(closes)) & (pos < len(closes)) & (pos > 0)
    if ok.sum() == 0:
        continue
    dates_ok = pd.to_datetime(s.date.values[ok])
    life.append(pd.DataFrame({
        'age_years': (dates_ok - first).days / 365.25,
        'price_pct': (closes[pos[ok]-1] - lo) / rng,
        'sent'     : s[SIG].values[ok],
        'fwd'      : (closes[ex[ok]] - closes[pos[ok]]) / closes[pos[ok]]}))

lf = pd.concat(life, ignore_index=True)
lf['correct'] = ((lf.sent > 0) & (lf.fwd > 0)) | ((lf.sent < 0) & (lf.fwd < 0))

lf['age_band']   = pd.cut(lf.age_years, [0,3,8,15,100],
                          labels=['0–3 yrs','3–8 yrs','8–15 yrs','15+ yrs'])
lf['price_band'] = pd.cut(lf.price_pct, [0,0.25,0.60,0.85,1.01],
                          labels=['Near low','Mid-range','Upper','Near high'])

grid = (lf.dropna(subset=['age_band','price_band'])
        .groupby(['age_band','price_band'], observed=True)
        .agg(hit=('correct','mean'), n=('correct','size')).reset_index())
grid.loc[grid.n < 40, 'hit'] = np.nan
pivot = grid.pivot(index='age_band', columns='price_band', values='hit')
npiv  = grid.pivot(index='age_band', columns='price_band', values='n')
print(pivot.round(3).to_string())

price_band  Near low  Mid-range  Upper  Near high
age_band                                         
0–3 yrs        0.537      0.410  0.254        NaN
3–8 yrs        0.582      0.547  0.543      0.453
8–15 yrs       0.555      0.554  0.508      0.399
15+ yrs        0.616      0.619  0.523      0.499


In [23]:
z    = pivot.values.astype(float)
text = np.where(np.isnan(z), 'n/a', np.round(z*100, 0).astype('U8'))
text = np.where(np.isnan(z), 'n/a', np.char.add(np.char.rstrip(text, '.0'), '%'))

fig = go.Figure(go.Heatmap(
    z=z, x=[str(c) for c in pivot.columns], y=[str(i) for i in pivot.index],
    colorscale=[[0, NEG], [0.5, '#F2F2F2'], [1, POS]],
    zmid=0.5, zmin=float(np.nanmin(z)), zmax=float(np.nanmax(z)),
    text=text, texttemplate='%{text}',
    textfont=dict(size=17),
    customdata=npiv.values,
    hovertemplate='%{y}, %{x}<br>hit rate %{z:.1%}<br>%{customdata:,} signals<extra></extra>',
    colorbar=dict(title=dict(text='Hit rate', font=dict(size=12, color=MUTED)),
                  tickformat='.0%', thickness=14, len=0.7,
                  tickfont=dict(size=11, color=MUTED))))

fig.update_xaxes(title='Price position within the stock’s historical range', side='bottom')
fig.update_yaxes(title='Years since the company’s first trading day', autorange='reversed')

flat = grid.dropna(subset=['hit'])
bestc = flat.loc[flat.hit.idxmax()]
worstc = flat.loc[flat.hit.idxmin()]

titled(fig,
       f'Buzz is most predictive for {str(bestc.age_band).lower()} companies trading {str(bestc.price_band).lower()}',
       'Hit rate by company age and price position. Blue beats chance, orange trails it; '
       'cells with fewer than 40 signals are left blank',
       height=520)
fig.show()

print(f'Best  cell: {bestc.age_band} / {bestc.price_band} — {bestc.hit:.1%} on {bestc.n:,} signals')
print(f'Worst cell: {worstc.age_band} / {worstc.price_band} — {worstc.hit:.1%} on {worstc.n:,} signals')

Best  cell: 15+ yrs / Mid-range — 61.9% on 1,060 signals
Worst cell: 0–3 yrs / Upper — 25.4% on 67 signals


**Finding.** The signal is not uniformly useful across a company's life. The grid shows where crowd discussion still contains information the market has not absorbed, and where it does not — a practical filter, since both age and price position are known before any trade is placed.

---

## Question 10
### Would trading these signals have beaten the market, and when did it fail?

Everything so far measures whether the signal contains information. This asks whether that information survives contact with execution — holding periods, transaction costs, and the requirement to commit capital before knowing the outcome.

A single final number would hide the interesting part. The figure below pairs the equity curves with the rolling twelve-month margin over the S&P 500, so periods of underperformance are as visible as the headline result.

In [24]:
eq = equity.copy()
eq['date'] = pd.to_datetime(eq['date'])
eq = eq.set_index('date').sort_index()

SERIES = [
    ('S&P_500_SPY',     'S&P 500 (SPY)',   CONTEXT,   'dash',  2.0),
    ('Buy_&_Hold',      'Buy & Hold',      MUTED,     'solid', 2.0),
    ('Sector_Rotation', 'Sector Rotation', ACCENT,    'solid', 3.2),
    ('Position_Trader', 'Position Trader', HIGHLIGHT, 'solid', 2.2),
]
SERIES = [s for s in SERIES if s[0] in eq.columns]
curves = eq[[s[0] for s in SERIES]].ffill().dropna(how='all')

finals = {label: float(curves[col].dropna().iloc[-1]) for col, label, *_ in SERIES}
for k, v in sorted(finals.items(), key=lambda x: -x[1]):
    print(f'  {k:<20} ${v:>12,.0f}   {(v/10000-1)*100:>8.1f}%')

  Sector Rotation      $     141,694     1316.9%
  Position Trader      $      85,350      753.5%
  Buy & Hold           $      72,005      620.1%
  S&P 500 (SPY)        $      43,868      338.7%


In [25]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
    row_heights=[0.68, 0.32],
    subplot_titles=('', 'Rolling 12-month margin over the S&P 500'))

for col, label, colour, dash, width in SERIES:
    s = curves[col].dropna()
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, mode='lines', name=label,
        line=dict(color=colour, width=width, dash=dash),
        hovertemplate=f'<b>{label}</b><br>%{{x|%b %Y}}<br>$%{{y:,.0f}}<extra></extra>'),
        row=1, col=1)

# End-of-line labels replace legend lookup
for col, label, colour, dash, width in SERIES:
    s = curves[col].dropna()
    fig.add_annotation(x=s.index[-1], y=float(s.iloc[-1]),
                       text=f'  {label}  ${float(s.iloc[-1]):,.0f}',
                       showarrow=False, xanchor='left',
                       font=dict(size=12, color=colour), row=1, col=1)

if 'S&P_500_SPY' in curves.columns and 'Sector_Rotation' in curves.columns:
    r12 = curves.pct_change(252)
    margin = (r12['Sector_Rotation'] - r12['S&P_500_SPY']).dropna()
    fig.add_trace(go.Scatter(
        x=margin.index, y=margin.values, mode='lines',
        line=dict(color=ACCENT, width=2), name='Rotation minus SPY',
        showlegend=False,
        hovertemplate='%{x|%b %Y}<br>margin %{y:+.1%}<extra></extra>'),
        row=2, col=1)
    fig.add_trace(go.Scatter(
        x=margin.index, y=np.where(margin.values < 0, margin.values, 0),
        fill='tozeroy', mode='none',
        fillcolor='rgba(213,94,0,0.30)', showlegend=False,
        hoverinfo='skip'), row=2, col=1)
    fig.add_hline(y=0, line_color=MUTED, line_width=1.5, row=2, col=1)
    worst = margin.idxmin()
    fig.add_annotation(x=worst, y=float(margin.min()),
                       text=f'worst stretch<br>{float(margin.min()):+.0%}',
                       showarrow=True, arrowhead=0, arrowcolor=NEG, ax=0, ay=38,
                       font=dict(size=11, color=NEG), row=2, col=1)
    below = float((margin < 0).mean())
else:
    below = float('nan')

fig.update_yaxes(title='Portfolio value', tickprefix='$', row=1, col=1)
fig.update_yaxes(title='Margin', tickformat='+.0%', row=2, col=1)
fig.update_xaxes(title='', row=2, col=1)
fig.update_layout(showlegend=False, margin=dict(r=190))

best_label = max(finals, key=finals.get)
spy_val = finals.get('S&P 500 (SPY)', np.nan)
mult = finals[best_label] / spy_val if spy_val == spy_val else float('nan')

titled(fig,
       f'{best_label} turned $10,000 into ${finals[best_label]:,.0f}, about {mult:.1f}× the S&P 500',
       f'Starting capital $10,000 in 2015, including transaction costs. The lower panel '
       f'shows the strategy trailed the index roughly {below:.0%} of the time',
       height=760)
fig.show()

print(f'Sector Rotation trailed the S&P 500 in {below:.1%} of rolling 12-month windows.')
print('The margin panel is the honest part: outperformance is not continuous.')

Sector Rotation trailed the S&P 500 in 43.7% of rolling 12-month windows.
The margin panel is the honest part: outperformance is not continuous.


**Finding.** The signal is not uniformly useful across a company's life. The grid shows where crowd discussion still contains information the market has not absorbed, and where it does not — a practical filter, since both age and price position are known before any trade is placed.

---

# Conclusions

**Online chatter carries a real but modest edge.** Pooled across every ticker and signal day, sentiment calls direction correctly more often than a coin flip. The margin is small, as it must be — a large and obvious edge would already have been competed away.

**The most striking result is asymmetry.** Positive sentiment predicts gains at roughly 55%. Negative sentiment predicts correctly only about 45% of the time, which is not a weak signal but an *inverse* one: price tended to rise after negative chatter. Pessimism in public discussion looks closer to a contrarian buy indicator than a warning, which is a genuinely actionable finding and the strongest argument in this project for treating the two directions differently rather than as mirror images.

**Volume and accuracy are unrelated.** The feeds that publish most are not the ones most often right, and reliability barely improves as coverage grows. Weighting sources by volume — the intuitive choice — would hand influence to whichever feed is noisiest.

**The edge has decayed.** Accuracy was higher before 2021 than after, consistent with the retail-trading era bringing more participants who trade on the same public signals.

**It survives execution, but not comfortably.** The sentiment-driven strategies finish ahead of the passive benchmarks after costs, while spending long stretches behind the index along the way.

### Limitations

Sentiment is scored from headlines and post titles rather than full article text. VADER is a general-purpose model, not tuned for financial language where "beat expectations" and "missed estimates" carry meanings it does not know. Backtests exclude slippage and assume fills at the close. Coverage is heavily skewed toward a handful of large companies. Matching company names in free text produces false positives that survive filtering. Several results here rest on modest sample sizes once the data is split several ways, and the confidence bands on those figures should be read as part of the finding rather than decoration. Finally this is one historical period in one market, and the post-2021 decay is itself evidence that these relationships do not hold still.

### Interactive dashboard

A curated subset of these findings is deployed on Streamlit Community Cloud, with sector and date filters and a what-if calculator for arbitrary investment amounts.